In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import matplotlib.table as table

warnings.filterwarnings("ignore")

# ====================== 随机种子 ======================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 配置 checkpoint ======================
CLASS_CHECKPOINT = "best_Baseline.pt"         
REG_CHECKPOINT   = "best_dual_regression.pt"   

# ====================== 数据加载 ======================
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

# ====================== Dataset ======================
class AnalysisDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "text": self.texts[idx]
        }
        return item

# ====================== 分类模型 ======================
class ClassifierModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return logits

# ====================== 回归模型 ======================
class DualRegressionModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.dep_head = nn.Linear(hidden, 1)
        self.sui_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        dep_score = torch.sigmoid(self.dep_head(cls)).squeeze(-1)
        sui_score = torch.sigmoid(self.sui_head(cls)).squeeze(-1)
        return dep_score, sui_score

# ====================== 加载模型 ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class_model = ClassifierModel(MODEL_PATH).to(DEVICE)
class_model.load_state_dict(torch.load(CLASS_CHECKPOINT, map_location=DEVICE))
class_model.eval()

reg_model = DualRegressionModel(MODEL_PATH).to(DEVICE)
reg_model.load_state_dict(torch.load(REG_CHECKPOINT, map_location=DEVICE))
reg_model.eval()

test_dataset = AnalysisDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ====================== 推理 ======================
print("正在推理测试集...")
all_class_preds = []
all_dep_scores = []
all_sui_scores = []
all_true_labels = []
all_texts = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()
        texts = batch["text"]
        
        class_logits = class_model(input_ids, attention_mask)
        class_preds = torch.argmax(class_logits, dim=1).cpu().numpy()
        
        dep_score, sui_score = reg_model(input_ids, attention_mask)
        dep_score = dep_score.cpu().numpy()
        sui_score = sui_score.cpu().numpy()
        
        all_class_preds.extend(class_preds)
        all_dep_scores.extend(dep_score)
        all_sui_scores.extend(sui_score)
        all_true_labels.extend(labels)
        all_texts.extend(texts)

df_test = pd.DataFrame({
    "text": all_texts,
    "true_label": all_true_labels,
    "class_pred": all_class_preds,
    "dep_score": all_dep_scores,
    "sui_score": all_sui_scores
})

label_map = {0: 'Anxiety', 1: 'Depression', 2: 'Normal', 3: 'Suicidal'}
df_test["true_status"] = df_test["true_label"].map(label_map)
df_test["class_pred_status"] = df_test["class_pred"].map(label_map)

# ====================== 找典型样本 ======================
# 条件：真实 Suicidal (3)，分类判为 Depression (1)，回归 Sui score 高（>0.80）
typical_samples = df_test[
    (df_test["true_label"] == 3) &
    (df_test["class_pred"] == 1) &
    (df_test["sui_score"] > 0.80)
].copy()

print(f"\n找到 {len(typical_samples)} 个典型样本（Suicidal 被分类误判为 Depression，但回归 Sui score > 0.80）")

if len(typical_samples) == 0:
    print("未找到符合条件的样本，可降低 sui_score 阈值（如 0.75）或检查 checkpoint。")
else:
    # 截断文本，便于展示
    typical_samples['text_short'] = typical_samples['text'].apply(lambda x: x[:150] + '...' if len(x) > 150 else x)
    
    # 选前 5 个样本
    display_df = typical_samples.head(5)[['text_short', 'true_status', 'class_pred_status', 'dep_score', 'sui_score']]
    display_df.columns = ['Text (truncated)', 'True Label', 'Class Prediction', 'Dep Score', 'Sui Score']
    
    # 打印 Markdown 表格
    print("\n### 典型案例对比（Classification Error vs Regression High Risk）")
    print("| Text (truncated) | True Label | Class Prediction | Dep Score | Sui Score |")
    print("|------------------|------------|------------------|-----------|-----------|")
    for _, row in display_df.iterrows():
        print(f"| {row['Text (truncated)']} | {row['True Label']} | {row['Class Prediction']} | {row['Dep Score']:.3f} | {row['Sui Score']:.3f} |")
    


Using device: cuda


Loading weights: 100%|██| 197/197 [00:00<00:00, 418.84it/s, Materializing param=encoder.layer.11.output.dense.weight]
RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██| 197/197 [00:00<00:00, 496.36it/s, Materializing param=encoder.layer.11.o

正在推理测试集...


100%|██████████████████████████████████████████████████████████████████████████████| 233/233 [14:43<00:00,  3.79s/it]


找到 27 个典型样本（Suicidal 被分类误判为 Depression，但回归 Sui score > 0.80）

### 典型案例对比（Classification Error vs Regression High Risk）
| Text (truncated) | True Label | Class Prediction | Dep Score | Sui Score |
|------------------|------------|------------------|-----------|-----------|
| How much would I have to take to off myself. I have a lot of each. Would it be painful? alcohol and Ambien | Suicidal | Depression | 0.882 | 0.887 |
| To my single father who raised me for the past decade, I am sorry. I hope in the next life, you get a stronger daughter. You deserve better. I am sorr... | Suicidal | Depression | 0.852 | 0.808 |
| tired of living this lonely life | Suicidal | Depression | 0.847 | 0.804 |
| aj really not wwe champ anymore... I want to die | Suicidal | Depression | 0.857 | 0.829 |
| its just gone now cannot find the will to live | Suicidal | Depression | 0.872 | 0.877 |
